In [ ]:
"""
Notebook 1: Phase 2 Data Fetcher & Cache Builder
Run this on a standard CPU instance. It builds the dna_promoter_cache.npz file.
Features incremental saving and adaptive rate-limiting to prevent 429 crashes.
"""

import os
import json
import time
import urllib.request
import urllib.error
import concurrent.futures
from pathlib import Path
import numpy as np
import pandas as pd

# --- PATHS & CONSTANTS ---
REPO_ROOT = Path("/kaggle/input/datasets/ericdu847/kaggle-phase2-compositional-model-v1")
OUTPUT_DIR = Path("/kaggle/working")
DNA_CACHE_PATH = OUTPUT_DIR / "dna_promoter_cache.npz"

GENE_COORDS_PATH = REPO_ROOT / "gene_coords.csv"
SYM2ENSG_PATH = REPO_ROOT / "gene_symbol_to_ensg.csv"
DELTA_DIR = REPO_ROOT

GROUPS = ["B", "monocyte", "CD8T", "naive_CD4T"]
SEQ_LEN = 2000
UPSTREAM = 1500
DOWNSTREAM = 500

BASE_ONEHOT = {"A": [1, 0, 0, 0], "C": [0, 1, 0, 0], "G": [0, 0, 1, 0], "T": [0, 0, 0, 1]}

# --- WORKER FUNCTIONS ---
def fetch_promoter_dna(chrom, tss, strand, retries=5):
    start, end = (tss - UPSTREAM, tss + DOWNSTREAM) if strand == "+" else (tss - DOWNSTREAM, tss + UPSTREAM)
    start = max(1, start)
    url = f"https://api.genome.ucsc.edu/getData/sequence?genome=hg38;chrom={chrom};start={start};end={end}"
    
    backoff = 2.0
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(url, timeout=10) as resp:
                seq = json.loads(resp.read().decode()).get("dna", "").upper()
                if not seq: return None
                if strand == "-":
                    seq = seq.translate(str.maketrans("ACGT", "TGCA"))[::-1]
                return seq
        except urllib.error.HTTPError as e:
            if e.code == 429: # Too Many Requests
                print(f"\n[429] Throttled. Backing off for {backoff}s...")
                time.sleep(backoff)
                backoff *= 2.0
            else: return None
        except Exception:
            time.sleep(1.0)
    return None

def fetch_single_gene(gene, coords_lookup):
    if gene not in coords_lookup.index:
        return gene, np.zeros((SEQ_LEN, 4), dtype=np.float32)
    row = coords_lookup.loc[gene].iloc[0] if isinstance(coords_lookup.loc[gene], pd.DataFrame) else coords_lookup.loc[gene]
    seq = fetch_promoter_dna(row["chrom"], int(row["tss"]), row["strand"])
    
    arr = np.zeros((SEQ_LEN, 4), dtype=np.float32)
    if seq:
        for i, base in enumerate(seq[:SEQ_LEN]):
            if base in BASE_ONEHOT: arr[i] = BASE_ONEHOT[base]
    return gene, arr

# --- MAIN EXECUTION ---
def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # 1. Gather all required genes
    all_genes = pd.read_parquet(DELTA_DIR / f"scbloodnl_move1_delta_{GROUPS[0]}.parquet").index.tolist()

    # 2. Load existing cache to resume progress if restarted
    cache = {}
    if DNA_CACHE_PATH.exists():
        with np.load(DNA_CACHE_PATH) as npz:
            cache = {k: npz[k] for k in npz.files}
        print(f"Resuming: Loaded {len(cache)} genes from existing cache.")

    # 3. Load coordinates
    coords = pd.read_csv(GENE_COORDS_PATH)
    sym2ensg = pd.read_csv(SYM2ENSG_PATH)
    merged = sym2ensg.merge(coords, left_on="gene_id", right_on="gene")
    merged["tss"] = np.where(merged["strand"] == "+", merged["gene_start"], merged["gene_end"])
    coords_lookup = merged.set_index("symbol")[["chrom", "strand", "tss"]]

    missing_genes = [g for g in all_genes if g not in cache]
    print(f"Fetching {len(missing_genes)} missing genes...")

    # 4. Fetch missing in smaller parallel chunks
    chunk_size = 200
    for i in range(0, len(missing_genes), chunk_size):
        chunk = missing_genes[i:i+chunk_size]
        with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
            futures = {executor.submit(fetch_single_gene, g, coords_lookup): g for g in chunk}
            for future in concurrent.futures.as_completed(futures):
                gene, arr = future.result()
                cache[gene] = arr
                
        # Save incrementally after every chunk
        np.savez_compressed(DNA_CACHE_PATH, **cache)
        print(f"Progress: saved {len(cache)} / {len(all_genes)} genes to cache.")

    print("Data fetch completely finished!")

if __name__ == "__main__":
    main()